# Bayes Rule — 01 Intuition

This notebook builds the intuition for Bayes' Rule using counts, trees, and a reusable Python solver.

Main idea:

> Bayes' Rule is not just a formula. It is a way of updating belief after seeing evidence.

## 1. The Bayes Question

Most Bayesian problems ask something like this:

> Given that some evidence was observed, how likely is a hypothesis?

In symbols:

$$
P(H \mid E)
$$

Read this as:

> Probability of hypothesis \(H\), given evidence \(E\).

## 2. Bayes' Rule

$$
P(H \mid E) = \frac{P(E \mid H)P(H)}{P(E)}
$$

Where:

| Term | Meaning | Intuition |
|---|---|---|
| $P(H)$ | Prior probability | What did I believe before seeing any new evidence? |
| $P(E)$ | Marginal/Evidence Probability | How common is the Evidence overall? |
| $P(E \mid H)$ | Likelihood | How likely is the Evidence given the Hypothesis? |
| $P(H \mid E)$ | Posterior probability | What do I believe after seeing the evidence? |

The posterior is the updated probability after seeing the evidence.

## 3. General Bayesian Update

For multiple hypotheses:

$$
P(H_i \mid E) = \frac{P(E \mid H_i)P(H_i)}{\sum_j P(E \mid H_j)P(H_j)}
$$

## 4. Example: Resume Screening

A company uses an AI model to flag promising resumes.

Ground truth:

- 1% of applicants are truly exceptional.
- 99% are average.

Model behavior:

- It flags 80% of exceptional resumes.
- It wrongly flags 10% of average resumes.

Question:

> If a resume is flagged, what is the probability it is actually exceptional?

The answer is:

$$
P(\text{Exceptional} \mid \text{Flagged}) \approx 7.48\%
$$

This feels surprisingly low because exceptional resumes are rare. This is the **base-rate effect**.

## 5. Understanding With Counts

Use a population of 10,000 applicants.

In [3]:
population_size = 10_000
counts = {}

for h in priors:
    hypothesis_count = round(population_size * priors[h])
    evidence_count = round(hypothesis_count * likelihoods[h])
    no_evidence_count = hypothesis_count - evidence_count

    counts[h] = {
        "hypothesis_count": hypothesis_count,
        "evidence_count": evidence_count,
        "no_evidence_count": no_evidence_count,
    }

counts

{'Exceptional': {'hypothesis_count': 100,
  'evidence_count': 80,
  'no_evidence_count': 20},
 'Average': {'hypothesis_count': 9900,
  'evidence_count': 990,
  'no_evidence_count': 8910}}

In [4]:
total_flagged = sum(counts[h]["evidence_count"] for h in counts)
exceptional_and_flagged = counts["Exceptional"]["evidence_count"]
prob_exceptional_given_flagged = exceptional_and_flagged / total_flagged

print(f"Exceptional and flagged: {exceptional_and_flagged:,}")
print(f"Total flagged: {total_flagged:,}")
print(f"P(Exceptional | Flagged): {prob_exceptional_given_flagged:.2%}")

Exceptional and flagged: 80
Total flagged: 1,070
P(Exceptional | Flagged): 7.48%


So among 10,000 applicants:

- Exceptional applicants = 100
- Exceptional and flagged = 80
- Average applicants = 9,900
- Average but wrongly flagged = 990

Therefore:

$$
P(\text{Exceptional} \mid \text{Flagged})
= \frac{80}{80 + 990}
= 7.48\%
$$

## 6. Visualizing With Graphviz

In a probability tree:

- Nodes represent events/states and counts.
- Edges represent probabilities.
- Root-to-leaf paths multiply probabilities.
- Bayes' Rule compares the relevant evidence branches.

In [5]:
from graphviz import Digraph
from IPython.display import HTML

# Create graph
dot = Digraph(
    format="svg",
    graph_attr={
        "rankdir": "LR",   # Left → Right
        "ranksep": "1.2",  # Horizontal spacing between levels
        "nodesep": "0.7"   # Vertical spacing between nodes
    }
)

# Nodes
dot.node("S", "Start")
dot.node("H", "Hypothesis (H)")
dot.node("NH", "Not Hypothesis (¬H)")
dot.node("E1", "Evidence (E)")
dot.node("NE1", "No Evidence (¬E)")
dot.node("E2", "Evidence (E)")
dot.node("NE2", "No Evidence (¬E)")

# Edges
dot.edge("S", "H", label="Prior")
dot.edge("S", "NH", label="Complement Prior")

dot.edge("H", "E1", label="Likelihood")
dot.edge("H", "NE1", label="Complement")

dot.edge("NH", "E2", label="False Positive Rate")
dot.edge("NH", "NE2", label="True Negative Rate")

# Render SVG and make it responsive
svg = dot.pipe(format="svg").decode("utf-8")

# Force SVG to use the full available notebook width
svg = svg.replace(
    "<svg ",
    '<svg width="100%" height="auto" preserveAspectRatio="xMinYMin meet" '
)

HTML(f"""
<div style="width:100%; overflow-x:auto;">
    {svg}
</div>
""")

## 8. Main Takeaway

Bayes' Rule becomes intuitive when viewed as counts flowing through a tree:

1. Start with a population.
2. Split by prior probabilities.
3. Split each hypothesis by likelihoods.
4. Combine all branches where the evidence occurred.
5. The posterior is the fraction of the evidence group belonging to each hypothesis.

For this example:

$$
P(\text{Exceptional} \mid \text{Flagged})
= \frac{80}{80 + 990}
= 7.48\%
$$